# Data

In [ ]:
data_AE = []

count = 0



with open("/kaggle/input/base-line/publish_data.txt", "rb") as f:

    for line in f:

        temp = line.rstrip(b'\r\n').split(b'\x01')

        for i in range(len(temp)):

            temp[i] = temp[i].decode('utf-8')

        while len(temp) < 3:

            count += 1

            temp.append(b'NULL')

        if not data_AE or data_AE[-1][0] != temp[0]:

            data_AE.append([temp[0], {temp[1]: temp[2]}])

        else:

            data_AE[-1][1][temp[1]] = temp[2]



data_AE[0]

# Model

In [ ]:
from huggingface_hub import login

# Log in with your Hugging Face API token
login("")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model_id = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side='right')
tokenizer.pad_token = tokenizer.eos_token  # Set pad token as eos token
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

text_generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    pad_token_id=tokenizer.pad_token_id
)

In [ ]:
def extract_attributes(product_inputs, max_length=100):
    messages_template = [
        {"role": "system", "content": "You are a world-class algorithm for extracting information in structured formats."},
        {"role": "user", "content": """Extract the valid attribute values for the given attributes from the product title in JSON format.
              All valid attributes are provided in the schema. Unknown attribute values should be marked as 'NULL'.
              Respond with a JSON object. Do not generate any text other than the JSON object.

              Example 1:
              Title: 1pc 5L Large Capacity Collapsible Camping Folding Water Container Bucket Storage Water Bag Water Bottle Container Carrier
              Attributes: { "Size": "" } """},
        {"role": "assistant", "content": """{ "Size": "NULL" }"""},
        {"role": "user", "content": """Example 2:
              Title: ROCOTACTICAL Mens Tactical Pocket T-Shirts Breathable Coolmax Military Army Hiking Shirts Utility Anti UV Camping T-Shirts
              Attributes: { "Gender": "", "Material": "", "Brand Name": "" } """},
        {"role": "assistant", "content": """{ "Gender": "Men", "Material": "NULL", "Brand Name": "ROCOTACTICAL" }"""},
        {"role": "user", "content": """Example 3:
              Title: Newborn Kids Baby Girls Bathing Suits Cute 2PCS Flower Swimwear Bikini Set Swimsuit Bathing Suit
              Attributes: { "Model Number": "", "Gender": "", "Sport Type": "" } """},
        {"role": "assistant", "content": """{ "Model Number": "NULL", "Gender": "Girls", "Sport Type": "Swim" }"""},
    ]

    outputs = []
    for idx, inp in enumerate(product_inputs):
        if (idx+1)%500==0:
            print(f"Processing {idx+1}th example.")
            
        messages = messages_template.copy()
        messages.append({"role": "user", "content": inp})
        result = text_generator(
            messages,
            max_new_tokens=max_length,
            truncation=True
        )
        generated_text = result[0]["generated_text"][-1]["content"]
        outputs.append(generated_text)
    return outputs

# Inference

In [ ]:
import re
import json

def prompt_baseline(data):
    prompt_data = []
    for title, schema in data:
        schema_input = json.dumps({key: "" for key in schema.keys()})
        prompt = f"Title: {title}\nAttributes:{schema_input}"
        prompt_data.append(prompt)
    return prompt_data

In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset into training and testing sets

train_data, test_data = train_test_split(data_AE, test_size=0.3, random_state=42)

#Pre-process data to get prompts

prompts = prompt_baseline(test_data)

#Get results from baseline model

outputs = extract_attributes(prompts)

In [ ]:
# Save the final predicted values as a JSON file
with open('/kaggle/working/predicted_values_ae.json', 'w') as json_file:
    json.dump(outputs, json_file, indent=4)

# Evaluation

In [1]:
from typing import List, Union

from collections import Counter



class EvaluationMetrics:



    @staticmethod

    def exact_match(ground_truth: Union[str, List[str]], predicted: Union[str, List[str]]) -> float:

        if isinstance(ground_truth, list) and isinstance(predicted, list):

            scores = [1.0 if g == p else 0.0 for g, p in zip(ground_truth, predicted)]

            return sum(scores) / len(scores) if scores else 0.0

        return 1.0 if ground_truth == predicted else 0.0



    @staticmethod

    def bow_precision(ground_truth: Union[str, List[str]], predicted: Union[str, List[str]]) -> float:

        def single_precision(gt: str, pred: str) -> float:

            gt_words = Counter(gt.split())

            pred_words = Counter(pred.split())

            common = sum(min(gt_words[word], pred_words[word]) for word in pred_words)

            return common / sum(pred_words.values()) if pred_words else 0.0



        if isinstance(ground_truth, list) and isinstance(predicted, list):

            scores = [single_precision(g, p) for g, p in zip(ground_truth, predicted)]

            return sum(scores) / len(scores) if scores else 0.0

        return single_precision(ground_truth, predicted)



    @staticmethod

    def bow_recall(ground_truth: Union[str, List[str]], predicted: Union[str, List[str]]) -> float:

        def single_recall(gt: str, pred: str) -> float:

            gt_words = Counter(gt.split())

            pred_words = Counter(pred.split())

            common = sum(min(gt_words[word], pred_words[word]) for word in gt_words)

            return common / sum(gt_words.values()) if gt_words else 0.0



        if isinstance(ground_truth, list) and isinstance(predicted, list):

            scores = [single_recall(g, p) for g, p in zip(ground_truth, predicted)]

            return sum(scores) / len(scores) if scores else 0.0

        return single_recall(ground_truth, predicted)



    @staticmethod

    def bow_f1(ground_truth: Union[str, List[str]], predicted: Union[str, List[str]]) -> float:

        precision = EvaluationMetrics.bow_precision(ground_truth, predicted)

        recall = EvaluationMetrics.bow_recall(ground_truth, predicted)

        if precision + recall == 0:

            return 0.0

        return 2 * (precision * recall) / (precision + recall)

In [2]:
metrics = EvaluationMetrics()



ground_truth = "the cat on the mat"

predicted = "the cat is on the mat"



# Exact match

print(metrics.exact_match(ground_truth, predicted))  # Output: 0.0



# Bag of Words Precision

print(metrics.bow_precision(ground_truth, predicted))  # Output: 0.833



# Bag of Words Recall

print(metrics.bow_recall(ground_truth, predicted))  # Output: 1



# Bag of Words F1 score

print(metrics.bow_f1(ground_truth, predicted))

0.0
0.8333333333333334
1.0
0.9090909090909091


In [3]:
from sklearn.model_selection import train_test_split

data_AE = []

count = 0



with open("Data/publish_data.txt", "rb") as f:

    for line in f:

        temp = line.rstrip(b'\r\n').split(b'\x01')

        for i in range(len(temp)):

            temp[i] = temp[i].decode('utf-8')

        while len(temp) < 3:

            count += 1

            temp.append(b'NULL')

        if not data_AE or data_AE[-1][0] != temp[0]:

            data_AE.append([temp[0], {temp[1]: temp[2]}])

        else:

            data_AE[-1][1][temp[1]] = temp[2]


# Split dataset into training and testing sets

train_data, test_data = train_test_split(data_AE, test_size=0.3, random_state=42)

test_data[0]

['New 3-Dial Zinc Alloy Trigger Password Lock Gun Key for Firearms Pistol Air Hunting Accessories 2018 s',
 {'Model Number': 'Alloy Trigger Password Lock Gun Key'}]

In [4]:
import ast
import json

def load_json(file_path):
    """Load JSON from a file."""
    with open(file_path, 'r') as file:
        return json.load(file)

def merge_json_files(file_paths):
    """Merge multiple JSON files."""
    merged_data = []
    for path in file_paths:
        data = load_json(path)
        merged_data += data
    return [ast.literal_eval(s) for s in merged_data]

# List of JSON files to merge
json_files = ['baseline-outputs/First.json',
              'baseline-outputs/Second.json',
              'baseline-outputs/Third.json']

# Merge the JSON files
merged_data = merge_json_files(json_files)

merged_data[0]

{'Model Number': '2018'}

In [5]:
def evaluate_extracted_data(data, extracted_json_outputs):
    ground_truths, predictions = [], []

    for expected_data, extracted_data in zip(data, extracted_json_outputs):
        expected_keys = expected_data[1].keys()
        for key in expected_keys:
            ground_truths.append(expected_data[1][key])
            if extracted_data is None:
                predictions.append("NULL")
            else:
                predictions.append(str(extracted_data.get(key, "NULL")))

    evaluation_results = {
        "exact_match": metrics.exact_match(ground_truths, predictions),
        "bow_precision": metrics.bow_precision(ground_truths, predictions),
        "bow_recall": metrics.bow_recall(ground_truths, predictions),
        "bow_f1": metrics.bow_f1(ground_truths, predictions),
    }

    return evaluation_results

In [6]:
results = evaluate_extracted_data(test_data, merged_data)
print(results)

{'exact_match': 0.5185140331224076, 'bow_precision': 0.5969695086872421, 'bow_recall': 0.5949969472532978, 'bow_f1': 0.5959815957940492}
